In [7]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, List
from langgraph.graph.message import add_messages

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,

)

from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig

from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from dotenv import load_dotenv

load_dotenv()

# ======================
# LLM
# ======================
llm = ChatOpenAI(model="gpt-4o-mini")


# ======================
# TOOLS
# ======================

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


@tool
def get_weather(city: str) -> str:
    """Get weather for a given city"""
    return f"The weather in {city} is sunny ☀️"


tools = [multiply, get_weather]

llm_with_tools = llm.bind_tools(tools)


# ======================
# STATE
# ======================

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


# ======================
# NODES
# ======================

def agent_node(state: AgentState):
    """LLM decides what to do (chat or call tool)"""
    
    response = llm_with_tools.invoke(state["messages"])
    
    return {
        "messages": [response]
    }


tool_node = ToolNode(tools)


# ======================
# ROUTER (Decision Logic)
# ======================

def should_continue(state: AgentState):
    """Decide whether to call tool or end"""
    
    last_message = state["messages"][-1]

    # If the LLM wants to call a tool
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tool"

    return END


# ======================
# GRAPH
# ======================

graph = StateGraph(AgentState)

# Nodes
graph.add_node("agent", agent_node)
graph.add_node("tool", tool_node)

# Flow
graph.add_edge(START, "agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tool": "tool",
        END: END
    }
)

# After tool runs → go back to agent
graph.add_edge("tool", "agent")


# ======================
# MEMORY (IMPORTANT)
# ======================

memory = MemorySaver()

app = graph.compile(checkpointer=memory)



In [8]:
app

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [6]:

# ======================
# RUN
# ======================

config: RunnableConfig = {
    "configurable": {
        "thread_id": "user-1"  # This enables memory persistence
    }
}

# First interaction
response = app.invoke(
    {
        "messages": [HumanMessage(content="What is 5 multiplied by 6?")]
    },
    config=config
)

print("\n=== RESPONSE 1 ===")
print(response["messages"][-1].content)


# Second interaction (memory retained)
response2 = app.invoke(
    {
        "messages": [HumanMessage(content="Now multiply that result by 10")]
    },
    config=config
)

print("\n=== RESPONSE 2 ===")
print(response2["messages"][-1].content)



# third interaction (memory retained)
response2 = app.invoke(
    {
        "messages": [HumanMessage(content="What is the weather in Nigeria")]
    },
    config=config
)

print("\n=== RESPONSE 3 ===")
print(response2["messages"][-1].content)


=== RESPONSE 1 ===
5 multiplied by 6 is 30.

=== RESPONSE 2 ===
The result of multiplying 30 by 10 is 300.

=== RESPONSE 3 ===
The weather in Nigeria is sunny. ☀️
